# OpenWebUI RAG Pre-Embedding
Embeds a local file using OpenAI `text-embedding-3-small` and inserts vectors directly into the OpenWebUI PostgreSQL database.

## 1. Configuration

In [14]:
# --- Fill these in ---
OPENAI_API_KEY  = "sk-..."
DB_URL          = "postgresql://postgres:<PASSWORD>@<HOST>:<PORT>/railway"
FILE_PATH       = "/Users/munirdin/Documents/projects/aramco-mini-llm/aramco_knowledge_base_clean.txt"
FILE_ID         = "c26d73cb-ae48-4a4e-aca7-d0b6ee97ffba"   # copy from Section 2 output
COLLECTION_NAME = "2bad0dc7-35d7-48fc-bd2c-7b42ccb2189f"   # copy collection_name from Section 2 output
                                                            # this is the Knowledge collection ID, NOT file-{FILE_ID}

# --- Tuneable, safe to leave as-is ---
EMBEDDING_MODEL = "text-embedding-3-small"
VECTOR_DIM      = 1536    # must match PGVECTOR_INITIALIZE_MAX_VECTOR_LENGTH in Railway
CHUNK_SIZE      = 800     # tokens per chunk
CHUNK_OVERLAP   = 100     # token overlap between chunks
BATCH_SIZE      = 100     # embeddings per API call (max 2048)

In [15]:
import psycopg2
try:
    conn = psycopg2.connect(DB_URL, connect_timeout=10)
    print("Connected!", conn.get_dsn_parameters())
    conn.close()
except Exception as e:
    print(f"Failed: {e}")


Connected! {'user': 'postgres', 'channel_binding': 'prefer', 'connect_timeout': '10', 'dbname': 'railway', 'host': 'thomas.proxy.rlwy.net', 'port': '15444', 'options': '', 'sslmode': 'prefer', 'sslnegotiation': 'postgres', 'sslcompression': '0', 'sslcertmode': 'allow', 'sslsni': '1', 'ssl_min_protocol_version': 'TLSv1.2', 'gssencmode': 'prefer', 'krbsrvname': 'postgres', 'gssdelegation': '0', 'target_session_attrs': 'any', 'load_balance_hosts': 'disable'}


## 2. Find FILE_ID and COLLECTION_NAME from the database
1. Upload the file in OpenWebUI
2. Add it to a **Knowledge collection** (Workspace → Knowledge)
3. Run this cell — copy `FILE_ID` and `collection_name` into the config above

> If `collection_name` shows `(none)`, the file hasn't been added to a Knowledge collection yet. Do that first — otherwise RAG searches the wrong collection.

In [16]:
import psycopg2

conn = psycopg2.connect(DB_URL)
cur = conn.cursor()
cur.execute("""
    SELECT id, filename, created_at,
           meta->>'collection_name' AS collection_name
    FROM "file"
    ORDER BY created_at DESC LIMIT 10
""")
rows = cur.fetchall()
cur.close()
conn.close()

print(f"{'FILE_ID':<38} {'Filename':<38} {'collection_name':<38} {'Created'}")
print("-" * 130)
for row in rows:
    cname = str(row[3]) if row[3] else "(none — add to Knowledge first)"
    print(f"{str(row[0]):<38} {str(row[1]):<38} {cname:<38} {row[2]}")

print("\n→ Copy FILE_ID and collection_name into the config cell above.")

FILE_ID                                Filename                               collection_name                        Created
----------------------------------------------------------------------------------------------------------------------------------
fec44a83-8351-4e04-b19e-1ce90ae261f2   c26d73cb-ae48-4a4e-aca7-d0b6ee97ffba_aramco_knowledge_base_clean.txt 2bad0dc7-35d7-48fc-bd2c-7b42ccb2189f   1781560157
0536a268-4836-4acb-a1ac-e00bae2b4bc7   aramco_knowledge_base_clean.txt        (none — add to Knowledge first)        1781557073
c26d73cb-ae48-4a4e-aca7-d0b6ee97ffba   aramco_knowledge_base_clean.txt        (none — add to Knowledge first)        1781556711

→ Copy FILE_ID and collection_name into the config cell above.


## 3. Load and chunk the file

In [17]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# Read file — handles .txt and basic .pdf extraction
import os
ext = os.path.splitext(FILE_PATH)[1].lower()

if ext == ".pdf":
    try:
        import pypdf
    except ImportError:
        import subprocess; subprocess.run(["pip", "install", "pypdf"], check=True)
        import pypdf
    reader = pypdf.PdfReader(FILE_PATH)
    raw_text = "\n".join(page.extract_text() or "" for page in reader.pages)
else:
    with open(FILE_PATH, "r", encoding="utf-8", errors="ignore") as f:
        raw_text = f.read()

print(f"File loaded: {len(raw_text):,} characters")

# Chunk by tokens
tokens = enc.encode(raw_text)
print(f"Total tokens: {len(tokens):,}")

chunks = []
start = 0
while start < len(tokens):
    end = min(start + CHUNK_SIZE, len(tokens))
    chunk_tokens = tokens[start:end]
    chunk_text = enc.decode(chunk_tokens).strip()
    if chunk_text:
        chunks.append(chunk_text)
    start += CHUNK_SIZE - CHUNK_OVERLAP

print(f"Chunks created: {len(chunks)}")

File loaded: 10,212,405 characters
Total tokens: 2,663,718
Chunks created: 3806


## 4. Embed with OpenAI

In [18]:
import time
import json
import os
from openai import OpenAI, RateLimitError
from tqdm import tqdm
import tiktoken

client = OpenAI(api_key=OPENAI_API_KEY)
enc = tiktoken.get_encoding("cl100k_base")

PROGRESS_FILE = "embedding_progress.json"
TPM_LIMIT = 900_000  # stay 10% under the 1M hard limit

# Load saved progress if resuming
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        saved = json.load(f)
    all_embeddings = saved["embeddings"]
    start_batch = saved["next_batch"]
    print(f"Resuming from batch {start_batch} ({len(all_embeddings)} embeddings already done)")
else:
    all_embeddings = []
    start_batch = 0

# Sliding window to track tokens used in the last 60 seconds
token_window = []  # list of (timestamp, token_count)

def tokens_used_last_minute():
    cutoff = time.time() - 60
    while token_window and token_window[0][0] < cutoff:
        token_window.pop(0)
    return sum(t for _, t in token_window)

batch_indices = list(range(start_batch * BATCH_SIZE, len(chunks), BATCH_SIZE))

for i in tqdm(batch_indices, desc="Embedding", initial=start_batch, total=(len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    batch_tokens = sum(len(enc.encode(c)) for c in batch)

    # Proactive throttle: wait if adding this batch would exceed TPM limit
    while tokens_used_last_minute() + batch_tokens > TPM_LIMIT:
        time.sleep(0.5)

    # Retry loop with exponential backoff
    delay = 5
    for attempt in range(8):
        try:
            response = client.embeddings.create(input=batch, model=EMBEDDING_MODEL)
            break
        except RateLimitError:
            if attempt == 7:
                raise
            print(f"\nRate limited, waiting {delay}s (attempt {attempt+1}/8)...")
            time.sleep(delay)
            delay = min(delay * 2, 120)

    all_embeddings.extend([e.embedding for e in response.data])
    token_window.append((time.time(), batch_tokens))

    # Save progress after each batch
    batch_num = i // BATCH_SIZE + 1
    with open(PROGRESS_FILE, "w") as pf:
        json.dump({"embeddings": all_embeddings, "next_batch": batch_num}, pf)

# Clean up progress file on success
if os.path.exists(PROGRESS_FILE):
    os.remove(PROGRESS_FILE)

print(f"Embeddings done: {len(all_embeddings)} vectors of dim {len(all_embeddings[0])}")


Embedding: 100%|██████████| 39/39 [03:24<00:00,  5.24s/it]

Embeddings done: 3806 vectors of dim 1536


## 5. Insert into PostgreSQL

In [19]:
import uuid
import json
import psycopg2

conn = psycopg2.connect(DB_URL)
cur = conn.cursor()

# Clear any existing (stuck) vectors for this collection first
cur.execute(
    "DELETE FROM document_chunk WHERE collection_name = %s",
    (COLLECTION_NAME,)
)
print(f"Cleared existing chunks for '{COLLECTION_NAME}'")

# Insert new vectors
for i, (chunk, emb) in enumerate(zip(chunks, all_embeddings)):
    cur.execute("""
        INSERT INTO document_chunk (id, collection_name, vector, text, vmetadata)
        VALUES (%s, %s, %s::vector, %s, %s::jsonb)
        ON CONFLICT (id) DO NOTHING
    """, (
        str(uuid.uuid4()),
        COLLECTION_NAME,
        str(emb),
        chunk,
        json.dumps({"file_id": FILE_ID, "page": i, "source": os.path.basename(FILE_PATH)})
    ))

conn.commit()
cur.close()
conn.close()

print(f"Inserted {len(chunks)} chunks into '{COLLECTION_NAME}'")

Cleared existing chunks for '2bad0dc7-35d7-48fc-bd2c-7b42ccb2189f'
Inserted 3806 chunks into '2bad0dc7-35d7-48fc-bd2c-7b42ccb2189f'


## 6. Verify

In [20]:
conn = psycopg2.connect(DB_URL)
cur = conn.cursor()
cur.execute(
    "SELECT COUNT(*) FROM document_chunk WHERE collection_name = %s",
    (COLLECTION_NAME,)
)
count = cur.fetchone()[0]
cur.close()
conn.close()

print(f"Chunks in DB for '{COLLECTION_NAME}': {count}")
print("Done — go test RAG in OpenWebUI (use #KnowledgeName in chat).")

Chunks in DB for '2bad0dc7-35d7-48fc-bd2c-7b42ccb2189f': 3806
Done — go test RAG in OpenWebUI (use #KnowledgeName in chat).


## Important: set these Railway env vars to match
```
RAG_EMBEDDING_ENGINE=openai
RAG_EMBEDDING_MODEL=text-embedding-3-small
OPENAI_API_KEY=sk-...
```
OpenWebUI uses the same model to embed the user's query at search time — it must match what you used here.

In [21]:
import psycopg2
import json

conn = psycopg2.connect(DB_URL)
cur = conn.cursor()
cur.execute('SELECT id, filename, data, meta FROM "file" ORDER BY created_at DESC LIMIT 5')
rows = cur.fetchall()
cur.close()
conn.close()

for row in rows:
    print(f"ID: {row[0]}")
    print(f"Filename: {row[1]}")
    if row[2]:
        d = dict(row[2])
        if "content" in d:
            d["content"] = d["content"][:100] + "..."
        print(f"Data: {json.dumps(d, indent=2)}")
    print(f"Meta: {json.dumps(row[3], indent=2) if row[3] else None}")
    print("-" * 60)


ID: fec44a83-8351-4e04-b19e-1ce90ae261f2
Filename: c26d73cb-ae48-4a4e-aca7-d0b6ee97ffba_aramco_knowledge_base_clean.txt
Data: {
  "status": "completed",
  "content": "SAUDI ARAMCO \u2014 COMPLETE KNOWLEDGE BASE\nGenerated from: Official Annual Reports (2017-2026), Quarterl..."
}
Meta: {
  "name": "c26d73cb-ae48-4a4e-aca7-d0b6ee97ffba_aramco_knowledge_base_clean.txt",
  "content_type": "text/plain",
  "size": 549,
  "file_hash": "1c74cd8acbf7f1d387292226d832116614d485b0dcd10c9a7fa910f25a947c46",
  "data": {
    "knowledge_id": "2bad0dc7-35d7-48fc-bd2c-7b42ccb2189f",
    "directory_id": null
  },
  "collection_name": "2bad0dc7-35d7-48fc-bd2c-7b42ccb2189f"
}
------------------------------------------------------------
ID: 0536a268-4836-4acb-a1ac-e00bae2b4bc7
Filename: aramco_knowledge_base_clean.txt
Data: {
  "error": "[Errno None] Can not write request body for https://api.openai.com/v1/embeddings",
  "status": "completed",
  "content": "SAUDI ARAMCO \u2014 COMPLETE KNOWLEDGE BASE\nGener

In [22]:
# Run this to see if chunks are in the DB
conn = psycopg2.connect(DB_URL)
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM document_chunk WHERE collection_name = %s", (COLLECTION_NAME,))
print(cur.fetchone()[0])  # should be 3806


3806


In [23]:
print(len(all_embeddings))  # should be 3806


3806


In [26]:
"""
title: Aramco Knowledge Base RAG
author: Munirdin Jadikar
description: Retrieves relevant chunks from the Aramco knowledge base using pgvector similarity search.
required_open_webui_version: 0.3.0
requirements: psycopg2-binary, openai
"""

from pydantic import BaseModel, Field
from typing import Callable, Any
import json


class Tools:
    class Valves(BaseModel):
        OPENAI_API_KEY: str = Field(
            default="",
            description="OpenAI API key (used to embed the query)",
        )
        DB_URL: str = Field(
            default="postgresql://postgres:<PASSWORD>@<HOST>:<PORT>/railway",
            description="PostgreSQL connection string (Railway)",
        )
        COLLECTION_NAME: str = Field(
            default="2bad0dc7-35d7-48fc-bd2c-7b42ccb2189f",
            description="OpenWebUI knowledge collection UUID",
        )
        EMBEDDING_MODEL: str = Field(
            default="text-embedding-3-small",
            description="OpenAI embedding model — must match what was used to build the index",
        )
        TOP_K: int = Field(
            default=6,
            description="Number of chunks to return",
        )

    def __init__(self):
        self.valves = self.Valves()

    def search_knowledge_base(self, query: str) -> str:
        """
        Search the Saudi Aramco knowledge base for information relevant to the query.
        Use this whenever the user asks about Aramco financials, operations, strategy,
        annual reports, production figures, or any Aramco-specific data.
        :param query: The user's question or search terms.
        :return: Relevant excerpts from the Aramco knowledge base.
        """
        try:
            from openai import OpenAI
            import psycopg2

            # 1. Embed the query
            client = OpenAI(api_key=self.valves.OPENAI_API_KEY)
            response = client.embeddings.create(
                input=[query],
                model=self.valves.EMBEDDING_MODEL,
            )
            query_vector = response.data[0].embedding
            vector_str = "[" + ",".join(str(x) for x in query_vector) + "]"

            # 2. Query pgvector — cosine similarity (lower <=> distance = more similar)
            conn = psycopg2.connect(self.valves.DB_URL, connect_timeout=10)
            cur = conn.cursor()
            cur.execute(
                """
                SELECT text,
                       1 - (vector <=> %s::vector) AS similarity,
                       vmetadata
                FROM document_chunk
                WHERE collection_name = %s
                ORDER BY vector <=> %s::vector
                LIMIT %s
                """,
                (vector_str, self.valves.COLLECTION_NAME, vector_str, self.valves.TOP_K),
            )
            rows = cur.fetchall()
            cur.close()
            conn.close()

            if not rows:
                return "No relevant information found in the Aramco knowledge base."

            # 3. Format results
            parts = [f"### Aramco Knowledge Base — top '{len(rows)}' results for: '{query}'\n"]
            for i, (text, similarity, meta) in enumerate(rows, 1):
                page = ""
                if meta and isinstance(meta, dict) and "page" in meta:
                    page = f" (chunk {meta['page']})"
                parts.append(f"**[{i}]** (similarity: {similarity:.3f}){page}\n{text.strip()}\n")

            return "\n---\n".join(parts)

        except Exception as e:
            return f"RAG search failed: {e}"


In [27]:

import sys; sys.path.insert(0, '.')
from aramco_rag_tool import Tools

t = Tools()
t.valves.OPENAI_API_KEY =OPENAI_API_KEY
print(t.search_knowledge_base('What was Abqaiq facility?'))



### Aramco Knowledge Base — top 4 results for: 'What was Abqaiq facility?'

---
**[1]** (similarity: 0.330) (chunk 292)
on Aramco’s
existing, transfer pricing regulations or operations and reputation.
the imposition of new restrictions on
foreign trade, investment or travel;
• adverse changes in economic and trade
sanctions, import or export controls
and national security measures
resulting in business disruptions,
including delays or denials of import or
export licences or blocked or rejected
financial transactions;

100
In February 2019, members of the U.S. defendant in a climate change case
House of Representatives and the U.S. brought by the Rhode Island Attorney
3 Senate introduced a bill that sought to General against 21 oil and gas companies
make unlawful certain conduct by foreign (“Rhode Island”). The defendants initially
states, state instrumentalities and state attempted to remove the case to federal
agents, such as taking action collectively court, but the case was remanded